# Clase 12 — Market making — Intro

El otro lado del mercado: en vez de cruzar, cotizas bid y ask y ganas el spread. Pero acumulas inventario, y el inventario es riesgo. Skew por inventario como primera defensa.

**Hoy construyes:** MarketMaker: cotizar a ambos lados y gestionar inventario.

## Cómo usar este cuaderno

- **Núcleo (en clase):** ejercicios 1 a 3.
- **Si vamos bien:** ejercicios 4 en adelante.
- **Casa / auxiliares:** el cuaderno `*_auxiliary.ipynb`.

Inténtalo, ejecuta la comprobación (`assert`) y mira la solución solo si te atascas.

## 1. Cotiza alrededor del mid

**Practicas:** MarketMaker.quotes.

Crea un `MarketMaker('BTC', half_spread=0.5)` y un libro con mid 100. Guarda `bid` y `ask` de `quotes(book)`.

In [ ]:
from exchange import OrderBook, Level
from exchange.strategies import MarketMaker
book = OrderBook('BTC', [Level(99.5,1)], [Level(100.5,1)])
mm = None
bid = None
ask = None

In [ ]:
assert bid < 100 < ask, 'cotizas por debajo y por encima del mid'
assert abs((ask - bid) - 1.0) < 1e-9, 'spread = 2*half_spread'
print('ok  bid=%.2f ask=%.2f' % (bid, ask))

### Solución guiada

```python
from exchange import OrderBook, Level
from exchange.strategies import MarketMaker
mm = MarketMaker('BTC', half_spread=0.5)
bid, ask = mm.quotes(book)
```

## 2. El skew baja las cotizaciones si estás largo

**Practicas:** inventario -> reservation price.

Con el mismo MM, fija `mm._inventory = 2` y vuelve a cotizar. El nuevo `bid2` debe ser menor que el bid de inventario 0.

In [ ]:
from exchange import OrderBook, Level
from exchange.strategies import MarketMaker
book = OrderBook('BTC', [Level(99.5,1)], [Level(100.5,1)])
mm = MarketMaker('BTC', half_spread=0.5, inventory_skew=1.0)
bid0, _ = mm.quotes(book)
bid2 = None

In [ ]:
assert bid2 < bid0, 'largo de inventario -> cotizas más bajo para soltar'
print('ok  bid0=%.2f bid2=%.2f' % (bid0, bid2))

### Solución guiada

```python
mm._inventory = 2
bid2, _ = mm.quotes(book)
```

## 3. Reservation price

**Practicas:** centro de las cotizaciones.

El reservation price con inventario positivo debe estar por debajo del mid. Guarda `r` con `mm._inventory=2`, mid 100.

In [ ]:
from exchange.strategies import MarketMaker
mm = MarketMaker('BTC', inventory_skew=1.0)
mm._inventory = 2
r = None

In [ ]:
assert r < 100, 'largo -> reservation price por debajo del mid'
print('ok  r=%.2f' % r)

### Solución guiada

```python
r = mm.reservation_price(100)
```

## 4. Simula al market maker

**Practicas:** MMSimulation.

Corre `MMSimulation(MarketMaker('BTC', half_spread=0.3), steps=300)`. Guarda `pnl` y `max_inv` del resultado.

In [ ]:
from exchange.strategies import MarketMaker
from exchange.simulation import MMSimulation
pnl = None
max_inv = None

In [ ]:
assert isinstance(pnl, float) and max_inv >= 0
print('ok  pnl=%.2f max|inv|=%.2f' % (pnl, max_inv))

### Solución guiada

```python
res = MMSimulation(MarketMaker('BTC', half_spread=0.3), steps=300).run()
pnl = res.final_pnl
max_inv = res.max_inventory
```

## Cierre

El market maker gana el spread, pero su enemigo es el inventario: cotiza para volver a plano.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.